# 零售销售数据挖掘与用户分群分析

基于Kaggle超市销售数据集，完成数据清洗、多维度营收统计、用户画像分析与K-Means用户聚类。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import os

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 创建输出目录
os.makedirs('../img', exist_ok=True)

## 1. 数据清洗

处理缺失值、异常值，清理约7.1%无效数据。

In [ ]:
# 读取数据
df = pd.read_csv('../data/supermarket_sales.csv')
print(f'原始数据行数: {len(df)}')
print(df.head())

# 检查缺失值
print('\n缺失值统计:')
print(df.isnull().sum())

# 处理缺失值
df_clean = df.dropna()
print(f'\n剔除缺失值后行数: {len(df_clean)}')

# 检查异常值 - Unit price
price_stats = df_clean['Unit price'].describe()
print('\n单价统计:')
print(price_stats)

# 过滤异常单价（低于1或高于200视为异常）
df_clean = df_clean[(df_clean['Unit price'] >= 1) & (df_clean['Unit price'] <= 200)]

# 过滤异常数量
df_clean = df_clean[(df_clean['Quantity'] >= 1) & (df_clean['Quantity'] <= 10)]

# 过滤异常评分
df_clean = df_clean[(df_clean['Rating'] >= 0) & (df_clean['Rating'] <= 10)]

# 计算清洗率
clean_rate = (len(df) - len(df_clean)) / len(df) * 100
print(f'\n清洗后数据行数: {len(df_clean)}')
print(f'无效数据占比: {clean_rate:.1f}%')

# 添加月份字段
df_clean['Date'] = pd.to_datetime(df_clean['Date'], dayfirst=True)
df_clean['Month'] = df_clean['Date'].dt.month

# 保存清洗后数据
df_clean.to_csv('../data/supermarket_sales_cleaned.csv', index=False)
print('\n清洗后数据已保存')

## 2. 多维度营收统计

从分店、产品品类、支付方式维度进行月度营收汇总。

In [ ]:
# 按分店统计营收
branch_revenue = df_clean.groupby('Branch')['Sales'].sum().sort_values(ascending=False)

# 按品类统计营收
category_revenue = df_clean.groupby('Product line')['Sales'].sum().sort_values(ascending=False)

# 按支付方式统计营收
payment_revenue = df_clean.groupby('Payment')['Sales'].sum().sort_values(ascending=False)

# 按月度统计营收
monthly_revenue = df_clean.groupby('Month')['Sales'].sum()

print('各分店营收:')
print(branch_revenue)

print('\n各品类营收:')
print(category_revenue)

print('\n各支付方式营收:')
print(payment_revenue)

print('\n月度营收:')
print(monthly_revenue)

## 3. 可视化图表

In [ ]:
# 分店营收柱状图
plt.figure(figsize=(10, 6))
branch_revenue.plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('各分店营收对比')
plt.xlabel('分店')
plt.ylabel('营收 (USD)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../img/branch_revenue.png', dpi=100)
plt.show()

In [ ]:
# 品类营收柱状图
plt.figure(figsize=(12, 6))
category_revenue.plot(kind='bar', color=['#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE'])
plt.title('各品类营收对比')
plt.xlabel('产品品类')
plt.ylabel('营收 (USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../img/category_revenue.png', dpi=100)
plt.show()

In [ ]:
# 支付方式营收饼图
plt.figure(figsize=(8, 8))
payment_revenue.plot(kind='pie', autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('支付方式营收占比')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../img/payment_revenue.png', dpi=100)
plt.show()

In [ ]:
# 月度营收趋势
plt.figure(figsize=(10, 6))
monthly_revenue.plot(kind='line', marker='o', color='#2C3E50', linewidth=2)
plt.title('月度营收趋势')
plt.xlabel('月份')
plt.ylabel('营收 (USD)')
plt.xticks([1, 2, 3])
plt.grid(True)
plt.tight_layout()
plt.savefig('../img/monthly_revenue.png', dpi=100)
plt.show()

## 4. 用户画像分析

会员vs非会员消费对比、男女客单价对比。

In [ ]:
# 会员vs非会员消费对比
member_stats = df_clean.groupby('Customer type')['Sales'].agg(['sum', 'mean', 'count'])
member_stats.columns = ['总消费', '客单价', '订单数']
print('会员vs非会员消费统计:')
print(member_stats)

# 男女客单价对比
gender_stats = df_clean.groupby('Gender')['Sales'].agg(['sum', 'mean', 'count'])
gender_stats.columns = ['总消费', '客单价', '订单数']
print('\n男女消费统计:')
print(gender_stats)

In [ ]:
# 会员vs非会员客单价对比
plt.figure(figsize=(8, 6))
member_stats['客单价'].plot(kind='bar', color=['#3498DB', '#E74C3C'])
plt.title('会员与非会员客单价对比')
plt.xlabel('客户类型')
plt.ylabel('客单价 (USD)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../img/member_vs_normal.png', dpi=100)
plt.show()

In [ ]:
# 男女客单价对比
plt.figure(figsize=(8, 6))
gender_stats['客单价'].plot(kind='bar', color=['#FF69B4', '#4169E1'])
plt.title('男女客单价对比')
plt.xlabel('性别')
plt.ylabel('客单价 (USD)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../img/gender_comparison.png', dpi=100)
plt.show()

## 5. K-Means用户聚类

依据消费总额、购买频次划分为高价值/普通/低频3类客群。

In [ ]:
# 构建用户特征
user_features = df_clean.groupby('Invoice ID').agg({
    'Sales': 'sum',           # 消费总额
    'Quantity': 'sum',        # 购买数量
    'Rating': 'mean'          # 平均评分
}).reset_index()

print('用户特征数据:')
print(user_features.head())

# 标准化特征
scaler = StandardScaler()
features_scaled = scaler.fit_transform(user_features[['Sales', 'Quantity']])

# K-Means聚类
kmeans = KMeans(n_clusters=3, random_state=42)
user_features['Cluster'] = kmeans.fit_predict(features_scaled)

# 查看聚类中心
print('\n聚类中心:')
print(scaler.inverse_transform(kmeans.cluster_centers_))

# 各类别统计
cluster_stats = user_features.groupby('Cluster').agg({
    'Sales': ['mean', 'sum', 'count'],
    'Quantity': ['mean', 'sum'],
    'Rating': 'mean'
})
print('\n各类别统计:')
print(cluster_stats)

# 为聚类结果命名
cluster_names = {
    0: '普通客户',
    1: '高价值客户',
    2: '低频客户'
}

# 根据消费总额确定类别名称
cluster_means = user_features.groupby('Cluster')['Sales'].mean()
sorted_clusters = cluster_means.sort_values(ascending=False).index.tolist()
cluster_names = {
    sorted_clusters[0]: '高价值客户',
    sorted_clusters[1]: '普通客户',
    sorted_clusters[2]: '低频客户'
}
user_features['Cluster_Name'] = user_features['Cluster'].map(cluster_names)

print('\n各类别命名:')
print(user_features['Cluster_Name'].value_counts())

In [ ]:
# 聚类结果可视化
plt.figure(figsize=(12, 8))
colors = ['#E74C3C', '#3498DB', '#2ECC71']
markers = ['o', 's', '^']

for cluster in range(3):
    subset = user_features[user_features['Cluster'] == cluster]
    plt.scatter(subset['Sales'], subset['Quantity'], 
                c=colors[cluster], marker=markers[cluster],
                label=cluster_names[cluster], alpha=0.7, s=100)

# 绘制聚类中心
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='*', s=300, label='聚类中心')

plt.title('K-Means用户聚类结果')
plt.xlabel('消费总额 (USD)')
plt.ylabel('购买数量')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../img/kmeans_clusters.png', dpi=100)
plt.show()

In [ ]:
# 各类别占比饼图
cluster_counts = user_features['Cluster_Name'].value_counts()

plt.figure(figsize=(8, 8))
cluster_counts.plot(kind='pie', autopct='%1.1f%%', colors=['#E74C3C', '#3498DB', '#2ECC71'])
plt.title('客户群体分布')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../img/cluster_distribution.png', dpi=100)
plt.show()

## 6. 经营优化建议

In [ ]:
# 输出分析报告
print('=' * 60)
print('零售销售数据挖掘分析报告')
print('=' * 60)

# 数据概览
print('\n【数据概览】')
print(f'原始数据: {len(df)} 条')
print(f'清洗后数据: {len(df_clean)} 条')
print(f'无效数据占比: {clean_rate:.1f}%')

# 营收分析
print('\n【营收分析】')
print(f'总营收: {df_clean["Sales"].sum():,.2f} USD')
print(f'平均客单价: {df_clean["Sales"].mean():,.2f} USD')
print(f'最高营收分店: {branch_revenue.idxmax()} ({branch_revenue.max():,.2f} USD)')
print(f'最高营收品类: {category_revenue.idxmax()} ({category_revenue.max():,.2f} USD)')

# 用户分析
print('\n【用户分析】')
print(f'会员占比: {len(df_clean[df_clean["Customer type"] == "Member"]) / len(df_clean) * 100:.1f}%')
print(f'会员客单价: {member_stats.loc["Member", "客单价"]:,.2f} USD')
print(f'非会员客单价: {member_stats.loc["Normal", "客单价"]:,.2f} USD')

# 聚类分析
print('\n【聚类分析】')
cluster_summary = user_features.groupby('Cluster_Name').agg({
    'Sales': ['mean', 'sum', 'count']
})
print(cluster_summary)

# 优化建议
print('\n【经营优化建议】')
print('1. 会员营销: 会员客单价高于非会员，建议推出更多会员专属优惠')
print('2. 品类优化: 根据各品类营收调整库存，重点关注高营收品类')
print('3. 客户分层运营:')
print('   - 高价值客户: 提供VIP服务，增加忠诚度')
print('   - 普通客户: 通过促销活动提升购买频次')
print('   - 低频客户: 发送个性化推荐，激活沉睡用户')
print('4. 支付方式: 支持多种支付方式，优化支付体验')

print('\n分析完成！可视化图表已保存至 img 目录。')